In [0]:
# newwwwwwwwwwwwwwwwww



from pyspark.sql.functions import col, window, sum as _sum, avg, count, current_timestamp

# ---------------------------------------------------------
# 1. NEW PATH CONFIGURATION
# ---------------------------------------------------------
SILVER_VOLUME_PATH = "/Volumes/proj_databricks/uber_medallion_layers/uber_silver"
GOLD_VOLUME_PATH   = "/Volumes/proj_databricks/uber_medallion_layers/uber_gold_realtime"

# ---------------------------------------------------------
# 2. READ STREAMS FROM UBER_SILVER
# ---------------------------------------------------------
df_driver = spark.readStream.format("delta").load(f"{SILVER_VOLUME_PATH}/silver_driver_locations")
df_ride   = spark.readStream.format("delta").load(f"{SILVER_VOLUME_PATH}/silver_ride_events")

# ---------------------------------------------------------
# 3. BUILD GOLD STAR SCHEMA TABLES
# ---------------------------------------------------------

# TABLE 1: DIM_DRIVER (City, Speed & Driver Status)
dim_driver = (df_driver
    .groupBy("driver_id", "city", "driver_status")
    .agg(
        count("*").alias("total_pings"),
        avg("latitude").alias("last_latitude"),
        avg("longitude").alias("last_longitude")
    )
    .withColumn("last_updated", current_timestamp())
)

# TABLE 2: DIM_CUSTOMER (Customer Spending & Rides)
dim_customer = (df_ride
    .groupBy("customer_id")
    .agg(
        count("ride_id").alias("total_rides_taken"),
        _sum("fare_amount").alias("total_lifetime_spent"),
        avg("fare_amount").alias("avg_fare_paid"),
        avg("distance_km").alias("avg_distance_traveled")
    )
    .withColumn("last_updated", current_timestamp())
)

# TABLE 3: FACT_RIDE_EVENTS (Granular Fact Table)
fact_ride_events = (df_ride
    .select(
        col("ride_id"),
        col("driver_id"),
        col("customer_id"),
        col("pickup_city"),
        col("drop_city"),
        col("ride_status"),
        col("payment_method"),
        col("fare_amount"),
        col("distance_km"),
        col("event_timestamp"),
        current_timestamp().alias("gold_processed_time")
    )
)

# TABLE 4: FACT_REVENUE_SUMMARY (City & Payment Summary)
fact_revenue_summary = (df_ride
    .withWatermark("event_timestamp", "10 minutes")
    .groupBy(
        window(col("event_timestamp"), "5 minutes"),
        col("pickup_city"),
        col("payment_method"),
        col("ride_status")
    )
    .agg(
        _sum("fare_amount").alias("total_revenue"),
        avg("fare_amount").alias("avg_fare"),
        avg("distance_km").alias("avg_distance"),
        count("ride_id").alias("total_rides")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("pickup_city"),
        col("payment_method"),
        col("ride_status"),
        col("total_revenue"),
        col("avg_fare"),
        col("avg_distance"),
        col("total_rides"),
        current_timestamp().alias("calculated_at")
    )
)

# ---------------------------------------------------------
# 4. WRITE STREAMS TO UBER_GOLD LOCATION
# ---------------------------------------------------------
q1 = dim_driver.writeStream.format("delta").outputMode("complete") \
    .trigger(processingTime='10 seconds') \
    .option("checkpointLocation", f"{GOLD_VOLUME_PATH}/_checkpoints/dim_driver") \
    .start(f"{GOLD_VOLUME_PATH}/dim_driver")

q2 = dim_customer.writeStream.format("delta").outputMode("complete") \
    .trigger(processingTime='10 seconds') \
    .option("checkpointLocation", f"{GOLD_VOLUME_PATH}/_checkpoints/dim_passenger") \
    .start(f"{GOLD_VOLUME_PATH}/dim_passenger")

q3 = fact_ride_events.writeStream.format("delta").outputMode("append") \
    .trigger(processingTime='10 seconds') \
    .option("checkpointLocation", f"{GOLD_VOLUME_PATH}/_checkpoints/fact_ride_events") \
    .start(f"{GOLD_VOLUME_PATH}/fact_ride_events")

q4 = fact_revenue_summary.writeStream.format("delta").outputMode("append") \
    .trigger(processingTime='10 seconds') \
    .option("checkpointLocation", f"{GOLD_VOLUME_PATH}/_checkpoints/fact_revenue_summary") \
    .start(f"{GOLD_VOLUME_PATH}/fact_revenue_summary")



In [0]:
# Create Metastore Views mapping to uber_gold location
spark.sql("CREATE SCHEMA IF NOT EXISTS proj_databricks.uber_medallion_layers")

spark.sql("CREATE OR REPLACE VIEW proj_databricks.uber_medallion_layers.dim_driver AS SELECT * FROM delta.`/Volumes/proj_databricks/uber_medallion_layers/uber_gold_realtime/dim_driver`")
spark.sql("CREATE OR REPLACE VIEW proj_databricks.uber_medallion_layers.dim_passenger AS SELECT * FROM delta.`/Volumes/proj_databricks/uber_medallion_layers/uber_gold_realtime/dim_passenger`")
spark.sql("CREATE OR REPLACE VIEW proj_databricks.uber_medallion_layers.fact_ride_events AS SELECT * FROM delta.`/Volumes/proj_databricks/uber_medallion_layers/uber_gold_realtime/fact_ride_events`")
spark.sql("CREATE OR REPLACE VIEW proj_databricks.uber_medallion_layers.fact_revenue_summary AS SELECT * FROM delta.`/Volumes/proj_databricks/uber_medallion_layers/uber_gold_realtime/fact_revenue_summary`")

print("🎉 ALL METASTORE VIEWS UPDATED TO UBER_GOLD!")

🎉 ALL METASTORE VIEWS UPDATED TO UBER_GOLD!


In [0]:
%sql
-- 1. Use your catalog
USE CATALOG proj_databricks;

-- 2. Drop Metastore Views first (jo Power BI ke liye banaye the)
DROP VIEW IF EXISTS proj_databricks.uber_medallion_layers.dim_driver;
DROP VIEW IF EXISTS proj_databricks.uber_medallion_layers.dim_passenger;
DROP VIEW IF EXISTS proj_databricks.uber_medallion_layers.fact_ride_events;
DROP VIEW IF EXISTS proj_databricks.uber_medallion_layers.fact_revenue_summary;

-- 3. Drop Gold Tables (agar managed/external tables register ki thi)
DROP TABLE IF EXISTS proj_databricks.uber_medallion_layers.dim_driver;
DROP TABLE IF EXISTS proj_databricks.uber_medallion_layers.dim_passenger;
DROP TABLE IF EXISTS proj_databricks.uber_medallion_layers.fact_ride_events;
DROP TABLE IF EXISTS proj_databricks.uber_medallion_layers.fact_revenue_summary;

-- 4. Drop Silver & Bronze Tables (agar catalog me register thi)
DROP TABLE IF EXISTS proj_databricks.uber_medallion_layers.silver_driver_locations;
DROP TABLE IF EXISTS proj_databricks.uber_medallion_layers.silver_ride_events;
DROP TABLE IF EXISTS proj_databricks.uber_medallion_layers.bronze_driver_locations;
DROP TABLE IF EXISTS proj_databricks.uber_medallion_layers.bronze_ride_events;

-- 5. (Optional) Drop Schema completely if you want a fresh schema setup
-- DROP SCHEMA IF EXISTS proj_databricks.uber_medallion_layers CASCADE;



In [0]:
# Clear physical files inside the volumes to remove old uneven data completely
dbutils.fs.rm("/Volumes/proj_databricks/uber_medallion_layers/bronze", True)
dbutils.fs.rm("/Volumes/proj_databricks/uber_medallion_layers/uber_silver", True)
dbutils.fs.rm("/Volumes/proj_databricks/uber_medallion_layers/uber_gold", True)

print("🗑️ All Volume files and checkpoints deleted!")

🗑️ All Volume files and checkpoints deleted!
